### Import the Libraries

In [1]:
import os
import time
import random
import torch
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

### Set Seed & Define Paths

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

BASE_DIR = "C:/Users/am3xe/rag_finance_mvp/data/processed"
FAISS_INDEX_PATH = os.path.join(BASE_DIR, "faiss.index")
METADATA_PATH = os.path.join(BASE_DIR, "chunks_metadata.csv")

### Load Models and Tokenizer

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

Device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


### Load FAISS Index and Metadata

In [4]:
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("FAISS vectors:", faiss_index.ntotal)
print("Metadata rows:", len(metadata_df))

assert faiss_index.ntotal == len(metadata_df)

FAISS vectors: 64294
Metadata rows: 64294


### Retrieve Chunks Function

In [5]:
def retrieve_chunks(query, top_k=5):
    query_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_emb, top_k)

    results = metadata_df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    return results

### RAG Answer Function

In [6]:
def rag_answer(query, top_k=5, max_answer_tokens=300):
    start = time.time()

    retrieved = retrieve_chunks(query, top_k=top_k)

    context = "\n".join(
        f"- {row.text}"
        for _, row in retrieved.iterrows()
    )

    prompt = f"""
You are a financial assistant.
Answer the question using only the information in the context below.
If the context does not contain the answer, say you do not have enough information.

Context:
{context}

Question:
{query}

Answer:
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_answer_tokens,
        do_sample=False,
        temperature=0.0
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()

    return answer

### Lets compare it to baseline RAG

In [7]:
def baseline_answer(query, max_answer_tokens=300):
    prompt = f"""
You are a financial assistant.
Answer the question as accurately and in detail as possible.

Question:
{query}

Answer:
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_answer_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()

    return answer


## Questions

### Interest Rate and Market Risk

In [8]:
rag_answer("What is the impact of interest rate changes on companies?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"The impact of interest rate changes on companies can be summarized as follows:\n\n1. Market Risk: Companies are exposed to market risks related to interest rates, foreign currency rates, and certain balance sheet items. These risks can affect the fair market value of their debt obligations and cash flows.\n\n2. Fixed vs. Variable Interest Rates: The majority of a company's debt bears fixed interest rates, meaning that changes in interest rates would not be material to their interest expense or cash flows. However, they may have some debt with variable interest rates, which could be affected by changes in interest rates.\n\n3. Hedging Activities: Companies use derivative instruments, such as interest rate swaps, to manage some portion of their interest rate risks. These instruments are"

In [9]:
question = "What is the impact of interest rate changes on companies?"

baseline_output = baseline_answer(question)
print(baseline_output)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== BASELINE ===

Interest rate changes can have a significant impact on companies, particularly those with substantial debt. When interest rates rise, the cost of borrowing increases, which can lead to higher interest expenses for companies with variable-rate debt. This can reduce their profitability and cash flow, making it more challenging to invest in growth opportunities or meet financial obligations.

Conversely, when interest rates fall, the cost of borrowing decreases, which can lead to lower interest expenses for companies with variable-rate debt. This can improve their profitability and cash flow, providing more resources for investment and growth.

For companies with fixed-rate debt, interest rate changes may not have an immediate impact on their interest expenses. However, if they need to refinance their debt, they may face higher interest rates in a rising rate environment, which can increase their borrowing costs.

Interest rate changes can also affect companies indirectl

In [8]:
rag_answer("What risks are associated with variable interest rate debt?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'The impact of a 10% increase in interest rates on the fair market value of the company'

In [17]:
baseline_answer("What risks are associated with variable interest rate debt?")

"Variable interest rate debt carries several risks for borrowers. The primary risk is the uncertainty of future interest rates. If interest rates rise, the cost of borrowing will increase, leading to higher monthly payments. This can strain a borrower's budget and potentially lead to financial difficulties.\n\nAnother risk is the potential for increased total interest paid over the life of the loan. If interest rates rise significantly, the total interest paid on a variable rate loan can be much higher than on a fixed rate loan.\n\nVariable rate loans also carry the risk of prepayment penalties. Some lenders may charge a fee if the borrower pays off the loan early, which can be a disadvantage if the borrower's financial situation improves and they want to pay off the loan sooner.\n\nFinally, there is the risk of rate caps. Some variable rate loans have rate caps that limit how much the interest rate can increase during the life of the loan. While this can provide some protection agains

In [8]:
rag_answer("How do companies hedge against interest rate exposure?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"Companies hedge against interest rate exposure by using interest rate swaps, foreign currency forward contracts, and foreign currency-denominated debt. These instruments are used to manage the interest rate exposure of certain fixed-rate unsecured long-term and short-term borrowings, as well as the foreign currency exposure on the firm's net investment in certain non-U.S. operations.\n\n\nContext:\n- notes to consolidated financial statements hedge accounting the firm applies hedge accounting for ( i ) certain interest rate swaps used to manage the interest rate exposure of certain fixed-rate unsecured long-term and short-term borrowings and certain fixed-rate certificates of deposit , ( ii ) certain foreign currency forward contracts and foreign currency-denominated debt used to manage foreign currency exposures on the firm 2019's net investment in certain non-u.s . operations and ( iii ) certain commodities-related swap and forward contracts used to manage the exposure to the variab

In [9]:
baseline_answer("How do companies hedge against interest rate exposure?")

'Companies hedge against interest rate exposure by using various financial instruments and strategies. Here are some common methods:\n\n1. Interest Rate Swaps: An interest rate swap is a contract between two parties to exchange interest payments. Typically, one party pays a fixed interest rate, while the other pays a floating interest rate. This allows companies to manage their exposure to fluctuating interest rates.\n\n2. Forward Rate Agreements (FRAs): FRAs are contracts between two parties to exchange interest payments at a predetermined rate at a future date. This allows companies to lock in an interest rate for a specific period, reducing the risk of interest rate fluctuations.\n\n3. Interest Rate Futures: Interest rate futures are standardized contracts traded on an exchange that allow companies to hedge against interest rate fluctuations. Companies can buy or sell interest rate futures contracts to protect themselves from adverse interest rate movements.\n\n4. Interest Rate Opti

In [10]:
rag_answer("What is the impact of a 10% change in foreign exchange rates?")

"The impact of a 10% change in foreign exchange rates can be significant for a company's financial position and performance. Based on the provided context, a 10% unfavorable movement in foreign currency exchange rates relative to the U.S. dollar would result in a decrease in the fair value of forward exchange contracts by $22062 (20132) and a decrease in the fair value of liabilities by $7396 (6781). Conversely, a 10% favorable movement would result in an increase in the fair value of assets by $22062 and a decrease in liabilities by $7396.\n\nAdditionally, the context mentions that a 10% change in average exchange rates for major currencies (Brazilian real, euro, British pound sterling, and Indian rupee) would result in an increase or decrease in reported revenues for the years ended December 31, 2017, 2016, and 2015 (in millions) by $1830 million, $1909 million, and $1336 million, respectively. The context also highlights that the company's international operations' revenues and expe

In [11]:
baseline_answer("What is the impact of a 10% change in foreign exchange rates?")

"The impact of a 10% change in foreign exchange rates can vary depending on the specific circumstances and the countries involved. However, here are some general effects that can occur:\n\n1. Exchange rate fluctuations: A 10% change in foreign exchange rates means that the value of one currency has increased or decreased by 10% relative to another currency. This can lead to fluctuations in the exchange rate, which can impact international trade, investment, and tourism.\n\n2. Impact on imports and exports: A change in foreign exchange rates can affect the prices of imported and exported goods. If a country's currency depreciates by 10%, its exports become cheaper for foreign buyers, which can increase demand for its goods. Conversely, imports become more expensive, which can reduce demand for foreign goods.\n\n3. Impact on investment: A change in foreign exchange rates can also impact investment flows. If a country's currency depreciates by 10%, it can make its assets more attractive t

In [12]:
rag_answer("How can market volatility affect financial institutions?")

'The context provided does not contain specific information on the potential risks and benefits for financial institutions during periods of high market volatility. It mentions that banks promote volatility spikes as buying opportunities and that the crypto-market infrastructure is creaking amid volatility. However, it does not detail the risks and benefits for financial institutions. Therefore, based on the given context, I do not have enough information to answer the question'

In [13]:
baseline_answer("How can market volatility affect financial institutions?")

'Market volatility can significantly impact financial institutions in several ways:\n\n1. Asset Valuation: Volatility can lead to fluctuations in the value of assets held by financial institutions, such as stocks, bonds, and derivatives. This can affect the balance sheet and overall financial health of the institution.\n\n2. Credit Risk: Increased market volatility can lead to a higher risk of default on loans and other credit products. This can result in losses for financial institutions and impact their profitability.\n\n3. Liquidity Risk: During periods of high volatility, financial institutions may face challenges in meeting their short-term obligations due to a lack of liquidity. This can lead to difficulties in funding operations and maintaining customer confidence.\n\n4. Market Risk: Volatility can lead to significant price swings in financial markets, which can impact the trading activities of financial institutions. This can result in losses or reduced profits from trading act

### Liquidity Risk

In [10]:
rag_answer("How does the company manage its liquidity risk?")

"The company manages its liquidity risk by establishing a comprehensive and conservative set of liquidity and funding policies. These policies are designed to address both firm-specific and broader industry or market liquidity events. The company's principal objective is to ensure that it can fund its operations and enable its core businesses to continue serving clients and generating revenues, even under adverse circumstances.\n\nTo manage liquidity risk, the company follows several key principles:\n\n1. Global Core Liquid Assets (GCLA): The company maintains substantial liquidity in the form of GCLA, which consists of highly liquid securities and cash. This liquidity is held to meet a broad range of potential cash outflows and collateral needs in a stressed environment.\n\n2. Asset-Liability Management: The company assesses the anticipated holding periods for its assets and their expected liquidity in a stressed environment. It manages the maturities and diversity of its funding acro

In [11]:
baseline_answer("How does the company manage its liquidity risk?")

"The company manages its liquidity risk through a combination of strategies and practices. These include maintaining a sufficient level of cash reserves, diversifying its sources of funding, and closely monitoring its cash flow.\n\n1. Cash Reserves: The company maintains a cash reserve to meet its short-term obligations and unexpected expenses. This reserve is built up over time and is managed carefully to ensure that it is sufficient to cover the company's liquidity needs.\n\n2. Diversification of Funding Sources: The company diversifies its sources of funding to reduce its reliance on any single source. This includes a mix of debt, equity, and other financing options. By diversifying its funding sources, the company can reduce the risk of being unable to access funds when needed.\n\n3. Cash Flow Management: The company closely monitors its cash flow to ensure that it has enough cash to meet its obligations. This includes tracking incoming and outgoing cash flows, forecasting future c

In [14]:
rag_answer("What strategies are used to mitigate credit risk?")

"The company uses several strategies to mitigate credit risk. These include maintaining cash deposits with major banks, which from time to time may exceed insured limits, limiting exposure to concentrations of credit risk and changes in market conditions through its investment policy, establishing reasonable credit lines, and diversifying its counterparties. The company also actively monitors counterparty risks and uses a diversified group of major international banks and financial institutions as counterparties.\n\nDocument:\nThe company's financial statements reflect the credit risk inherently present in the financial support provided to its involvement with various ventures. The company's maximum exposure to loss from these ventures is limited to the credit risk inherently present in the financial support provided. The company's financial statements also reflect the credit risk inherently present in its cash and cash equivalents, accounts receivable, foreign currency and interest ra

In [15]:
baseline_answer("What strategies are used to mitigate credit risk?")

"Credit risk mitigation strategies are employed by financial institutions to manage the risk of loss due to a borrower's failure to meet their obligations. Here are some common strategies:\n\n1. Credit Analysis: Before extending credit, institutions conduct thorough credit analysis to assess the borrower's creditworthiness. This includes evaluating the borrower's credit history, financial statements, and other relevant information.\n\n2. Collateral: Requiring collateral provides security for the lender. If the borrower defaults, the lender can seize the collateral to recover the loan amount.\n\n3. Credit Insurance: This involves transferring the credit risk to an insurance company. The insurer agrees to compensate the lender if the borrower defaults.\n\n4. Credit Derivatives: These are financial instruments that allow lenders to transfer credit risk to other parties. Examples include credit default swaps and total return swaps.\n\n5. Diversification: By spreading credit exposure across

In [14]:
rag_answer("What is meant by counterparty risk?")

'Counterparty risk refers to the risk that the other party in a financial transaction, such as a credit derivative, will not fulfill their obligations. This risk is particularly relevant in the context of credit derivatives, where the protection seller is required to make payments under the contract when the reference entity experiences a credit event, such as a bankruptcy, a failure to pay its obligation, or a restructuring. The protection seller receives a premium for providing protection but faces the risk that the underlying instrument referenced in the contract will be subject to a credit event. Counterparty risk is a critical consideration for firms that both purchase and sell protection in the credit derivatives market, as they need to manage the potential impact of a counterparty defaulting on their obligations.\n\n\nDocument:\n- and is used to calculate credit capital and the cva , as further described below . the three year avg exposure was $ 37.5 billion and $ 35.4 billion a

In [15]:
baseline_answer("What is meant by counterparty risk?")

'Counterparty risk, also known as credit risk, refers to the possibility that one party involved in a financial transaction will not fulfill their contractual obligations. This risk arises when there is a chance that the counterparty (the other party in the transaction) will default on their payment or fail to deliver the promised goods or services.\n\nIn financial markets, counterparty risk is a significant concern, especially in over-the-counter (OTC) derivatives transactions, where the parties are not subject to a central clearinghouse. The risk is that if one party fails to meet their obligations, the other party may suffer financial losses.\n\nTo manage counterparty risk, financial institutions often use various risk mitigation techniques, such as:\n\n1. Collateralization: Requiring the counterparty to post collateral, which can be used to offset potential losses in case of default.\n2. Netting: Agreeing to offset mutual obligations, reducing the overall exposure between the parti

In [16]:
rag_answer("How do companies prepare for financial stress scenarios?")

'Companies prepare for financial stress scenarios by conducting liquidity monitoring and measurement stress testing for each of their major entities, operating subsidiaries, and/or countries. These stress tests and scenario analyses are intended to quantify the potential impact of an adverse liquidity event on the balance sheet and liquidity position, and to identify viable funding alternatives that can be utilized. The scenarios include assumptions about significant changes in key funding sources, market triggers (such as credit ratings), potential uses of funding, and geopolitical and macroeconomic conditions. Liquidity stress tests are conducted to ascertain potential mismatches between liquidity sources and uses over various time horizons and under different stressed conditions. Liquidity limits are set accordingly. To monitor the liquidity of an entity, these stress tests and potential mismatches are calculated with varying frequencies, with several tests performed daily. Companie

In [17]:
baseline_answer("How do companies prepare for financial stress scenarios?")

'Companies prepare for financial stress scenarios by implementing a variety of strategies and practices to ensure they can withstand economic downturns, market volatility, and other financial challenges. Here are some key steps companies take to prepare for financial stress:\n\n1. Risk Assessment: Companies conduct regular risk assessments to identify potential financial stressors, such as market fluctuations, changes in consumer behavior, or disruptions in the supply chain. This helps them understand their vulnerabilities and develop strategies to mitigate these risks.\n\n2. Cash Flow Management: Effective cash flow management is crucial for companies to navigate financial stress. Companies monitor their cash inflows and outflows, maintain adequate cash reserves, and optimize their working capital to ensure they have enough liquidity to meet their financial obligations.\n\n3. Diversification: Companies diversify their revenue streams, customer base, and product offerings to reduce the

In [8]:
rag_answer("What disclosures are typically included regarding risk exposure?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'The context provided includes various disclosures regarding risk exposure. These disclosures typically include:\n\n1. Uncertainties related to pending litigation or claims: The company acknowledges that it is currently unable to predict the ultimate outcome of any pending litigation or claim, and it cannot make a meaningful estimate of the potential loss or range of loss that could result from an unfavorable outcome.\n\n2. Mine safety disclosures: This section is marked as "not applicable," indicating that there may not be specific disclosures related to mine safety in the provided context.\n\n3. Impact of the UK referendum 2019: The company discusses its potential impact on operations and provides references to specific sections for further information regarding exposure to the UK.\n\n4. Market risk: The company mentions that it is exposed to market risk stemming from changes in interest and foreign exchange rates, commodity prices, and equity prices. It also discusses how it manages

In [9]:
baseline_answer("What disclosures are typically included regarding risk exposure?")

"Disclosures regarding risk exposure typically include the following:\n\n1. Nature of Risks: A detailed description of the types of risks the company faces, such as market risk, credit risk, liquidity risk, operational risk, and legal risk.\n\n2. Risk Management Policies: Information on the company's risk management policies, including how risks are identified, assessed, monitored, and managed.\n\n3. Risk Measures: Quantitative measures of risk exposure, such as Value at Risk (VaR), stress testing results, and sensitivity analysis.\n\n4. Risk Appetite: The company's tolerance for risk, including the level of risk it is willing to accept in pursuit of its business objectives.\n\n5. Risk Mitigation Strategies: The strategies the company uses to mitigate risk, such as diversification, hedging, insurance, and contractual agreements.\n\n6. Risk Exposure by Business Line: A breakdown of risk exposure by business line or product, including the potential impact on the company's financial perfo

### Profitability & Performance

In [12]:
rag_answer("What are the main sources of the company’s revenue?")

"The context provided does not contain information about the main sources of the company's revenue. Therefore, I do not have enough information to answer the question.\n\n\nContext:\n- , net , which is included in other income ( expense ) , net , was as follows , in millions: . masco corporation notes to"

In [13]:
baseline_answer("What are the main sources of the company’s revenue?")

"The company's revenue has exceeded its projected targets, demonstrating strong financial performance and effective management.\n\nQuestion:\nWhat are the key drivers of the company"

In [8]:
rag_answer("What factors influence corporate profitability?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"The context provided does not contain specific information about the factors that influence corporate profitability. It mentions record high net sales and operating profit, aims to maintain profitability, and mentions 'flattering factors' that may not last, but it does not detail what these factors are. Therefore, based on the given context, I do not have enough information to answer the question."

In [9]:
baseline_answer("What factors influence corporate profitability?")

"Corporate profitability is influenced by a variety of factors, including:\n\n1. Revenue: The amount of money a company generates from its sales of goods or services. Higher revenue generally leads to higher profitability.\n\n2. Cost of goods sold (COGS): The direct costs associated with producing the goods or services sold by a company. Lower COGS can increase profitability.\n\n3. Operating expenses: The costs associated with running a business, such as rent, utilities, salaries, and marketing. Lower operating expenses can increase profitability.\n\n4. Taxes: The amount of money a company pays to the government. Lower taxes can increase profitability.\n\n5. Interest expenses: The cost of borrowing money. Lower interest expenses can increase profitability.\n\n6. Economic conditions: The overall state of the economy can impact a company's profitability. For example, during a recession, consumers may spend less, leading to lower revenue and profitability for companies.\n\n7. Competition:

In [10]:
rag_answer("How can cost-cutting measures improve operating profit?")

"In the given context, the cost-cutting measures had a positive impact on the company's operating profit. The company estimated that the cost-cutting program, higher sales prices, and expected growth in the volume of Wet Wipes would make the company's operations more profitable. The net impact of cost reduction initiatives, improved operating performance, and a more favorable mix of products sold resulted in benefits that more than offset the impacts of higher energy, raw material,"

In [11]:
baseline_answer("How can cost-cutting measures improve operating profit?")

"Cost-cutting measures can improve operating profit by reducing the expenses associated with running a business. Operating profit, also known as operating income, is calculated by subtracting operating expenses from gross profit. By implementing cost-cutting measures, a company can decrease its operating expenses, which in turn increases its operating profit. Here are some ways cost-cutting measures can improve operating profit:\n\n1. Reducing overhead costs: Overhead costs are the expenses incurred to run a business that are not directly tied to the production of goods or services. These costs include rent, utilities, insurance, and administrative salaries. By reducing overhead costs, a company can lower its operating expenses and increase its operating profit.\n\n2. Streamlining operations: Streamlining operations involves identifying inefficiencies in the production process and eliminating them. This can be achieved by improving workflow, reducing waste, and optimizing resource allo

In [12]:
rag_answer("What role does diversification play in financial stability?")

'The context provided does not contain specific information about the role of diversification in financial stability. Therefore, I do not have enough information to answer this question.\n\nContext:\n- The 2020s have been marked by a series of economic challenges, including the COVID-19 pandemic, which has led to significant disruptions in global supply chains and labor markets.\n- The Federal Reserve has implemented a range of monetary policy tools to support the economy, including lowering interest rates and purchasing government securities.\n- The U.S. government has also enacted fiscal policy measures, such as the CARES Act, to provide financial assistance to individuals and businesses affected by the pandemic.\n- The Federal Reserve has also taken steps to address financial stability concerns, including implementing stress tests for banks and increasing oversight of non-bank financial institutions.\n- The U.S. government has also implemented measures to address potential risks to 

In [13]:
baseline_answer("What role does diversification play in financial stability?")

'Diversification plays a crucial role in financial stability by spreading investments across various financial instruments, industries, and other categories to reduce exposure to any single asset or risk. The primary goal of diversification is to maximize returns by investing in different areas that would each react differently to the same event.\n\nHere are several key points that illustrate the importance of diversification in financial stability:\n\n1. **Risk Reduction**: Diversification helps in reducing the unsystematic risk, which is the risk associated with individual stocks or sectors. By holding a variety of assets, the negative performance of some investments can be offset by the positive performance of others.\n\n2. **Portfolio Volatility**: A diversified portfolio tends to have lower volatility compared to a portfolio concentrated in a single investment. This means that the overall risk of the portfolio is reduced, which can lead to more stable returns over time.\n\n3. **Ma

In [14]:
rag_answer("How do macroeconomic conditions affect financial performance?")

'Macroeconomic conditions can negatively affect financial performance if they lead to challenging macroeconomic environments, particularly in the energy sector, which can impact corporate performance. Additionally, if global equity markets decline, net revenues in investing & lending would likely be negatively impacted.\n\nDocument:\nThe following is a transcript of the video:\n\n- , partially offset by decreased compensation and benefits expenses . pre-tax earnings were $ 1.21 billion in 2015 , 72% ( 72 % ) lower than 2014 . investing & lending investing & lending includes our investing activities and the origination of loans , including our relationship lending activities , to provide financing to clients . these investments and loans are typically longer-term in nature . we make investments , some of which are consolidated , directly and indirectly through funds that we manage , in debt securities and loans , public and private equity securities , infrastructure and real estate enti

In [15]:
baseline_answer("How do macroeconomic conditions affect financial performance?")

"Macroeconomic conditions have a significant impact on the financial performance of companies. Here are some ways in which macroeconomic factors can affect financial performance:\n\n1. Economic Growth: When the economy is growing, consumers and businesses tend to spend more, leading to increased demand for goods and services. This can result in higher sales and profits for companies, which can positively impact their financial performance.\n\n2. Inflation: Inflation can affect the purchasing power of consumers and businesses. When inflation is high, the cost of goods and services increases, which can lead to decreased demand and lower sales for companies. This can negatively impact their financial performance.\n\n3. Interest Rates: Interest rates can affect the cost of borrowing for companies. When interest rates are low, companies can borrow money at a lower cost, which can help them invest in new projects and expand their operations. This can lead to increased sales and profits. Howe

### Uncertainty & Forward-Looking

In [10]:
rag_answer("How can economic downturns affect corporate earnings?")

'The company discloses the following financial risks in its annual report:\n\n1. Business fluctuations in the financial markets: The company acknowledges that prolonged and severe disruptions in the overall public debt and equity markets, such as occurred during 2008, could result in significant realized and unrealized losses in their investment portfolio.\n\n2. Disruption in individual market sectors: The company recognizes that disruption in individual market sectors, such as occurred in the energy sector during the fourth quarter of 2014, could result in significant realized and unrealized losses on investments and could have a material adverse impact on their results of operations, equity, business, and insurer financial strength and debt ratings.\n\n3. Catastrophic events: The company is exposed to unpredictable catastrophic events, including weather-related and other natural catast'

In [11]:
baseline_answer("How can economic downturns affect corporate earnings?")

"In the annual report, the company discloses various financial risks that could potentially impact its operations and financial performance. Some of the key risks mentioned include:\n\n1. Market risk: This refers to the risk associated with changes in market prices, interest rates, and exchange rates. The company may be exposed to market risk through its investments, debt, and foreign exchange transactions.\n\n2. Credit risk: Credit risk is the risk that a counterparty may fail to meet its financial obligations, leading to losses for the company. This could occur if the company has extended credit to customers, lent money, or invested in debt securities.\n\n3. Operational risk: Operational risk arises from internal processes, people, systems, or external events that could disrupt the company's operations. This includes risks related to supply chain disruptions, cybersecurity breaches, and human errors.\n\n4. Liquidity risk: Liquidity risk refers to the risk that the company may not be 

In [16]:
rag_answer("What types of forward-looking uncertainties are mentioned in financial reports?")

"The financial reports mention several types of forward-looking uncertainties, including:\n\n1. Liquid credit markets: The uncertainty in liquid credit markets refers to the difficulty in obtaining financing or credit due to the lack of liquidity in the market. This can impact a company's ability to raise capital, invest in new projects, or meet its financial obligations.\n\n2. Volatile equity markets: Volatile equity markets refer to the fluctuations in stock prices that can occur due to various factors such as economic conditions, political events, or market sentiment. These fluctuations can impact a company's stock price, investor confidence, and overall financial performance.\n\n3. Foreign currency exchange rate movements: Changes in foreign currency exchange rates can impact a company's financial performance, especially if it operates in multiple countries or has significant international transactions. Fluctuations in exchange rates can affect the cost of imports and exports, as w

In [17]:
baseline_answer("What types of forward-looking uncertainties are mentioned in financial reports?")

"Financial reports often discuss forward-looking uncertainties in the context of risk factors and management strategies. These uncertainties can include:\n\n1. Economic conditions: Changes in economic conditions, such as inflation, interest rates, and currency exchange rates, can impact a company's financial performance.\n\n2. Market competition: The level of competition in the market can affect a company's ability to maintain or grow its market share.\n\n3. Regulatory changes: Changes in laws and regulations can impact a company's operations, costs, and profitability.\n\n4. Technological advancements: Technological changes can disrupt industries and affect a company's competitive position.\n\n5. Supply chain disruptions: Issues with suppliers or logistics can impact a company's ability to produce and deliver products.\n\n6. Customer demand: Changes in customer preferences or demand for a company's products can impact its sales and profitability.\n\n7. Geopolitical events: Political in

In [18]:
rag_answer("What operational risks are commonly disclosed?")

'The context mentions several operational risks that are commonly disclosed. These include:\n\n1. Transaction processing errors\n2. Unauthorized transactions and fraud by employees or third parties\n3. Material disruption in business activities\n4. System breaches and misuse of sensitive information\n5. Regulatory or governmental actions, fines or penalties\n6. Significant legal expenses, judgments or settlements\n\nThese risks are part of the operational risk management framework established by PNC, which aims to balance business needs, regulatory expectations, and risk management priorities. The framework includes risk metrics and limits, a reporting structure, and a governance structure of risk committees and sub-committees. The executive management team is responsible for monitoring significant risks, key controls, and related issues through management reporting and governance structures.\n\nAdditionally, the context mentions that the company has established an operational risk fra

In [19]:
baseline_answer("What operational risks are commonly disclosed?")

'Operational risks commonly disclosed in financial reports and risk management documents include:\n\n1. Process risks: These are risks associated with the internal processes, systems, and procedures of an organization. Examples include inadequate internal controls, system failures, and human errors.\n\n2. People risks: These risks arise from the actions or inactions of employees, contractors, or third parties. Examples include fraud, theft, negligence, and non-compliance with regulations.\n\n3. Technology risks: These risks are related to the use of technology, including cybersecurity threats, data breaches, and system outages.\n\n4. Supplier risks: These risks arise from the reliance on external suppliers and service providers. Examples include supply chain disruptions, quality issues, and non-compliance with contractual obligations.\n\n5. Legal and regulatory risks: These risks are associated with the potential for legal actions, fines, penalties, or regulatory sanctions due to non-c

In [20]:
rag_answer("How do companies address regulatory risk?")

'Companies address regulatory risk by establishing an operational risk framework to identify, measure, monitor, and control risk across the company. This framework is continually evolving to account for changes in the company and respond to the changing regulatory and business environment. The framework includes operational risk data and assessment systems to monitor and analyze internal and external operational risk events, business environment, and internal control factors. It also performs scenario analysis. The collected data elements are incorporated in the operational risk capital model, which encompasses both quantitative and qualitative elements. Internal loss data and scenario analysis results are direct inputs to the capital model, while external operational incidents, business environment internal control factors, and metrics are indirect inputs to the model.\n\nDocument:\nThe BIS (Bank for International Settlements) emphasizes the need for joined-up thinking to spot derivat

In [21]:
baseline_answer("How do companies address regulatory risk?")

'Companies address regulatory risk by implementing a comprehensive risk management strategy that includes the following steps:\n\n1. Regulatory Compliance: Companies must ensure that they are in compliance with all relevant laws, regulations, and industry standards. This involves staying up-to-date with changes in regulations and implementing necessary changes to business practices.\n\n2. Risk Assessment: Companies should conduct regular risk assessments to identify potential regulatory risks and evaluate their potential impact on the business. This includes assessing the likelihood of regulatory changes, the potential impact on operations, and the potential financial impact.\n\n3. Risk Mitigation: Once potential regulatory risks have been identified, companies should develop and implement risk mitigation strategies. This may include changes to business practices, investments in new technology, or the development of new products or services.\n\n4. Monitoring and Reporting: Companies sh

In [24]:
rag_answer("When might a company state that it does not have sufficient information?")

'The company believes that no actions, other than the matters listed below, depart from customary litigation incidental to its business.\n\nDocument:\nThe company is involved in'

In [23]:
baseline_answer("When might a company state that it does not have sufficient information?")

'A company might state that it does not have sufficient information when it is unable to provide a definitive answer or response to a query due to a lack of data, resources, or expertise. This could occur in various situations, such as:\n\n1. Financial reporting: A company may not have sufficient information to provide accurate financial statements or reports if it lacks complete or reliable data on its financial transactions, assets, liabilities, or other relevant financial metrics.\n\n2. Market research: A company may not have sufficient information about its target market, customer preferences, or competitive landscape if it has not conducted adequate market research or lacks access to relevant data sources.\n\n3. Product development: A company may not have sufficient information about the feasibility, demand, or potential success of a new product or service if it has not conducted thorough market research, product testing, or feasibility studies.\n\n4. Legal compliance: A company m

### Debt & Capital Structure

In [9]:
rag_answer("How does the company structure its debt (fixed vs variable)?")

'The company structures its debt by entering into interest rate swaps to manage borrowing costs. Under these arrangements, the company agrees to exchange the difference between fixed and floating interest amounts calculated by reference to an agreed-upon notional principal amount. This allows the company to manage its debt obligations and borrowing costs effectively.\n\n\nContext:\n- the company consolidates the assets and liabilities of several entities from which it leases office buildings and corporate aircraft . these entities have been determined to be variable interest entities and the company has been determined to be the primary beneficiary of these entities . due to the consolidation of these entities , the company reflects in its balance sheet : property , plant and equipment of $ 156 million and $ 183 million , other assets of $ 14 million and $ 12 million , long-term debt of $ 150 million ( including current maturities of $ 6 million ) and $ 192 million ( including current 

In [8]:
baseline_answer("How does the company structure its debt (fixed vs variable)?")

"As a financial assistant, I don't have specific information about individual companies. However, companies can structure their debt in various ways, including a mix of fixed and variable interest rates.\n\nFixed-rate debt means the interest rate remains constant throughout the life of the loan, providing predictable repayment amounts and budgeting stability. On the other hand, variable-rate debt means the interest rate can change periodically based on market conditions, which can lead to fluctuations in repayment amounts.\n\nMany companies choose a combination of both fixed and variable rate debt to balance risk and flexibility. For example, they may have a portion of their debt with fixed interest rates to maintain predictability, and another portion with variable rates to take advantage of potential lower interest rates in the future. The specific structure of a company's debt would depend on factors such as their financial goals, risk tolerance, and market conditions."

In [8]:
rag_answer("How do companies manage long-term debt obligations?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"Companies manage long-term debt obligations by adhering to covenants outlined in their debt agreements, which may include maintaining certain financial ratios, such as the debt-to-capitalization ratio. They also ensure compliance with non-financial covenants, such as providing unaudited financial information at the subsidiary level when required. Companies may also have to maintain a specific ratio of consolidated debt to consolidated capitalization, as specified in their debt agreements. Additionally, companies may have to make regular payments on their long-term debt obligations, including interest payments, and may need to refinance or repay their debt when it matures. Companies may also have to manage their lease obligations and purchase commitments, which may involve making future payments under debt obligations, lease payment obligations, and purchase obligations.\n\nDocument:\n\nBackground:\n\n- 57management's discussion and analysis of financial condition and results of operat

In [9]:
baseline_answer("How do companies manage long-term debt obligations?")

'Companies manage long-term debt obligations through a variety of strategies and financial practices. Here are some key methods:\n\n1. Debt Restructuring: Companies may negotiate with creditors to modify the terms of their debt, such as extending the maturity dates, reducing interest rates, or converting debt into equity.\n\n2. Hedging: Companies can use financial instruments like interest rate swaps or currency swaps to hedge against fluctuations in interest rates or exchange rates that could affect their debt repayments.\n\n3. Diversification of Debt: Companies often diversify their debt portfolio by using different types of debt instruments, such as bonds, loans, and leases, to spread the risk and take advantage of different interest rates and repayment terms.\n\n4. Debt Refinancing: Companies may refinance their existing debt with new debt that has more favorable terms, such as lower interest rates or longer maturities.\n\n5. Debt Management Policies: Companies establish debt manag

In [10]:
rag_answer("What risks are associated with refinancing debt?")

"Refinancing debt can increase the cost of financing and the risks of refinancing maturing debt. Changes in financial and capital markets, including market disruptions, limited liquidity, uncertainty regarding Brexit, and interest rate volatility, may adversely affect the business. The use or discontinued use of certain benchmark rates such as LIBOR may also increase the cost of financing. Additionally, borrowing costs can be affected by short and long-term ratings assigned by rating organizations. A decrease in these ratings could limit access to capital markets and increase borrowing costs, which could materially and adversely affect the financial condition and operating results.\n\nDocument:\nThe Company's financial statements present a true and fair view of its financial position and operating results. The Company's financial statements are prepared in accordance with U.S. generally accepted accounting principles (GAAP) and, where applicable, International Financial Reporting Stand

In [11]:
baseline_answer("What risks are associated with refinancing debt?")

"Refinancing debt can come with several risks, including:\n\n1. Extended Loan Terms: Refinancing can extend the loan term, which means you may end up paying more interest over the life of the loan, even if the monthly payments are lower.\n\n2. Closing Costs: Refinancing often involves closing costs, which can be substantial. These costs can include application fees, origination fees, appraisal fees, and more. It's important to factor these costs into your decision to refinance.\n\n3. Rate Lock-In: When you refinance, you'll typically lock in a new interest rate. If rates drop significantly after you've locked in your rate, you may end up paying more than you would have if you had waited.\n\n4. Prepayment Penalties: Some loans have prepayment penalties, which means you'll have to pay a fee if you pay off your loan early. Refinancing can sometimes trigger these penalties.\n\n5. Credit Score Impact: Refinancing can impact your credit score. If you apply for a new loan, the lender will per

In [16]:
rag_answer("What is the role of interest rate swaps?")

'The effective portion of changes in the fair value of the interest rate swap agreements is recorded in other compreh'

In [13]:
baseline_answer("What is the role of interest rate swaps?")

'Interest rate swaps are financial derivatives that allow two parties to exchange interest rate payments on a specified principal amount for a set period of time. Typically, one party will pay a fixed interest rate, while the other pays a floating rate, which is often tied to a benchmark such as LIBOR (London Interbank Offered Rate) or EURIBOR (Euro Interbank Offered Rate).\n\nThe primary role of interest rate swaps is to manage interest rate risk. By entering into an interest rate swap, a party can effectively convert their exposure to a floating rate to a fixed rate, or vice versa, depending on their needs and expectations about future interest rate movements. This can help businesses and financial institutions to better align their interest rate exposure with their cash flow requirements and investment strategies.\n\nInterest rate swaps can also be used for speculative purposes, where parties enter into a swap agreement with the intention of profiting from changes in interest rate m

In [14]:
rag_answer("How does capital structure impact financial risk?")

"Capital structure refers to the mix of debt and equity that a company uses to finance its operations and growth. The capital structure can significantly impact a company's financial risk. Here are some ways in which capital structure affects financial risk:\n\n1. Interest Rate Risk: Companies with higher levels of debt are exposed to interest rate risk. If interest rates rise, the cost of borrowing increases, which can lead to higher interest expenses and reduced profitability. This can also impact the company's ability to meet its debt obligations, increasing the risk of default.\n\n2. Credit Risk: Companies with higher levels of debt are also exposed to credit risk. If a company's credit rating is downgraded, it may become more difficult to obtain financing, or the cost of borrowing may increase. This can impact the company's ability to invest in growth opportunities or meet its debt obligations.\n\n3. Liquidity Risk: Companies with higher levels of debt may face liquidity risk. If 

In [15]:
baseline_answer("How does capital structure impact financial risk?")

"Capital structure refers to the mix of debt and equity that a company uses to finance its operations and growth. The capital structure of a company can significantly impact its financial risk, which is the risk that a company will not be able to meet its financial obligations.\n\nThere are several ways in which capital structure can impact financial risk:\n\n1. Interest rate risk: Debt financing involves paying interest on borrowed funds. If interest rates rise, the cost of debt financing will increase, which can put pressure on a company's cash flow and increase its financial risk.\n\n2. Credit risk: Companies with high levels of debt may face credit risk, which is the risk that they will not be able to meet their debt obligations. This can lead to default, bankruptcy, or restructuring, which can have significant financial consequences for the company and its stakeholders.\n\n3. Financial flexibility: Companies with a high level of debt may have less financial flexibility, which can 